# Setup

In [3]:
# Setup
import json
import os
import sys
from datetime import datetime, timedelta
import time
from dotenv import load_dotenv, set_key
import random
from pathlib import Path
import logging

# hashing for signing
import hashlib
import hmac

# requests
import requests

# data munging
import pandas as pd
import numpy as np

# helper functions
from tiktok_api_helpers import *

## API Setup

In [4]:
new_access_token = False
use_refresh_token = False

In [7]:
if (new_access_token | use_refresh_token):

    # get new acces_token
    token_url = "https://auth.tiktok-shops.com/api/v2/token/get"

    if use_refresh_token:
        auth_code = os.environ.get("TIKTOK_REFRESH_TOKEN")
    else:
        auth_code = os.environ.get("TIKTOK_AUTH_CODE")
        
    params = {
        "app_key": app_key,
        "app_secret": app_secret,
        "auth_code": auth_code,
        "grant_type": "authorized_code",
    }
    
    response = requests.get(token_url, params=params, timeout=15)
    data = response.json()['data']
    access_token = data['access_token']
    print(data)

    # save tokens in .env file
    set_key(".env", "TIKTOK_ACCESS_TOKEN", data['access_token'])
    set_key(".env", "TIKTOK_REFRESH_TOKEN", data['refresh_token'])

else:
    access_token = os.environ.get("TIKTOK_ACCESS_TOKEN")
    refresh_token = os.environ.get("TIKTOK_REFRESH_TOKEN")

{'access_token': 'ROW_4BYZwwAAAABwUkDEHls0a5HWUXrwXMKTiTad7aZWscGF3YO04dC618tsNg5IyUju9wmyG_yDYWyswxMJmZDUW6fgblohBSVjId5xJjmY2T2ZL6RFLjEx8j22AnX4jTTiS8spO7v1mEk', 'access_token_expire_in': 1785069965, 'refresh_token': 'ROW_ui9aiQAAAAC1Bb7xpqxtsXbX4YggPBqPSJC_n2KwyzjtGQJjqeBiCNIWMJnsgJvu_b0UxC8CJy8', 'refresh_token_expire_in': 4905899553, 'open_id': 'cU4EZQAAAACOV4AEeERJl-yievH_HBssPYsgA5YV2AbEDsJwxBBAMQ', 'seller_name': 'Vitami', 'seller_base_region': 'PH', 'user_type': 0, 'granted_scopes': ['data.bestselling.public.read', 'seller.authorization.info', 'seller.affiliate_collaboration.read', 'seller.affiliate_collaboration.write', 'seller.creator_marketplace.read', 'seller.affiliate_messages.write']}


## Load Creator Data

In [5]:
# load all creators
df_creators_list = pd.read_excel("creators/all_creators_sorted.xlsx", sheet_name="LIST_CREATOR", usecols=[1, 2, 25])
df_creators = pd.read_csv(CONSOLIDATED_CSV)

df_creators_all = df_creators_list.merge(df_creators, how='left', on="username")

In [6]:
df_creators[df_creators.duplicated(subset=['creator_open_id','username'], keep=False)]

,avatar,avg_ec_live_uv,avg_ec_video_view_count,category_ids,creator_open_id,follower_count,gmv_range,nickname,selection_region,top_follower_demographics,username,gmv,video_gmv,live_gmv


In [7]:
# df_creators = df_creators[~(df_creators['creator_open_id'] == 'PC3CtQAAAACOV4AEeERJl-yievH_HBss9ntAFA4yBGy1CxaMuRamZw')]

In [25]:
df_creators_gmv = pd.read_csv("creators/creators_gmv_units_sold.csv").drop_duplicates()

In [26]:
df_creators_gmv['username']

0              jasmintrisha
1              thisismedane
2                   lyza_gc
3              jake_recosph
4       bewell_conyotindero
               ...         
1655     murang.gadgetphone
1656      cellphone.shop596
1657             gamesexyph
1658              bscocurve
1659            kheem_irish
Name: username, Length: 1619, dtype: object

In [27]:
df_creators_gmv['username'].isin(df_creators['username']).sum()/df_creators_gmv['username'].shape[0]

np.float64(0.6979617047560223)

In [28]:
df_creators_gmv['username'].isin(df_creators['username']).sum()

np.int64(1130)

In [29]:
df_creators_gmv.merge(df_creators_list[['username','batch_name']], how='left', on='username')['batch_name'].value_counts(dropna=False).sort_index()

batch_name
All_202606_00k-02k       172
All_202606_02k-04k        42
All_202606_04k-06k        65
All_202606_06k-08k        48
All_202606_08k-10k        30
All_202606_10k-12k        27
Health_202606_00k-02k    500
Health_202606_02k-04k    114
Health_202606_04k-06k     50
Health_202606_06k-08k     31
Health_202606_08k-10k     21
Health_202606_10k-12k     16
NaN                      503
Name: count, dtype: int64

In [34]:
df_creators_gmv_merged = df_creators_list[['username','batch_name']].merge(df_creators_gmv, how='left', on='username')

In [39]:
df_creators_gmv_merged.head(100)['creator_open_id'].notnull().sum()

np.int64(73)